# M06 — Clusters, labels, and stability under changed measurements

<!-- paper-first -->
### Begin with the paper question

**Read or revisit [PD03](../../curriculum/papers/design.md#pd03).** Use the assigned first-pass sections in the guide; if you already read them, return only to the relevant figure or claim. Do this before the technical explanation below.

**Motivation:** Could unstable measurements create unstable clusters even if the clustering algorithm is deterministic?

Write a two-sentence prediction and one thing you cannot yet explain. Ask your AI tutor to locate evidence in the supplied paper and distinguish it from inference. A paper link motivates this question; it does not mean the paper uses every method demonstrated here.

**After the experiment:** revisit your prediction in the [evidence ledger](../../curriculum/coursework/EVIDENCE_LEDGER.md). Explain one mechanism you now understand, cite a result from this notebook, and name a paper claim this exercise still cannot test. Keep a small demonstration distinct from a reproduction of the study.
<!-- /paper-first -->

**Original guided lab · 75–100 minutes.** Read 20 min, predict/code 35 min, failure investigation 20 min, explain and transfer 15 min. Run all cells in order in a fresh kernel. All executed data are synthetic unless explicitly stated. No network, GPU, or external dataset is required.

Clustering assigns observations to groups according to a representation and a similarity rule. The algorithm does not know whether groups correspond to disease subtypes, scanner differences, task states, or artifacts. Before interpreting a cluster biologically, specify what each row represents and which features and units determine distance. Even familiar algorithms can produce completely different groupings after a seemingly innocent unit change.

K-means minimizes within-cluster squared Euclidean distances around a chosen number of centers. It favors a particular geometry and requires K in advance. A Gaussian mixture describes observations through several Gaussian components and provides posterior membership probabilities. Those components need not be biological populations either; a flexible distribution may use several components to describe one skewed population. Density-based or spectral methods encode different assumptions rather than removing assumptions altogether.

The experiment creates two genuine separated groups in a weak-scale feature and adds a high-scale nuisance feature. Raw K-means tends to cluster the nuisance axis. Standardizing features changes the geometry and recovers the planted grouping more accurately in this constructed case. Standardization is not universally correct: if the nuisance variable is scientifically irrelevant, explicitly excluding or modeling it may be better than assigning it equal weight. Unit choices should follow a justified measurement model.

Cluster labels are arbitrary names. Swapping label zero and one leaves the partition unchanged, so direct equality of label arrays is a poor stability measure. Adjusted Rand index compares partitions without relying on label names and adjusts for chance. In this teaching example we can compare with planted labels; in a real exploratory dataset those labels are unknown. Resampling stability, external replication, and association with independently measured variables answer different questions.

We refit on bootstrap samples and compare predictions on the original observations. Here the scaler stays fixed, so stability is conditional on this chosen representation; assessing the entire workflow would also refit its learned preprocessing within each resample. That assesses one kind of fitting stability, not proof that the clusters are natural kinds. Repeated observations from one participant would require participant-level resampling. A high silhouette or stable partition can still describe site. Conversely, a continuum can be cut into reproducible clusters even when a continuous scientific model would be more appropriate.

Ask AI to show cluster sizes, feature summaries, and site distributions before naming groups. Keep the exploratory nature visible when many values of K or many embeddings have been tried. A colorful two-dimensional plot can hide distortion, and using its apparent islands to choose an analysis can compound selection effects. Document the complete representation-to-partition chain.

## Transformation contract

Participant feature matrix → scaling choice → fitted centers → arbitrary cluster labels. The label compresses the feature vector and loses within-cluster variation. Bootstrap refits assess partition stability under a specified resampling scheme.

## Ask your AI tutor

```text
Explain this notebook one transformation at a time.
Before each cell ask me to predict shapes, units, and a check.
Give edits in executable cells of at most 20 lines.
Keep the prescribed split, random seed, and tests intact.
Distinguish generated suggestions from executed results.
After the failure experiment, ask me to explain the mechanism.
```

In [1]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score,silhouette_score
rng=np.random.default_rng(406)
y=np.repeat([0,1],150)
X=np.column_stack([3*(2*y-1)+rng.normal(0,.35,300),rng.normal(0,30,300)])
raw=KMeans(2,n_init=10,random_state=1).fit_predict(X)
Z=StandardScaler().fit_transform(X)
model=KMeans(2,n_init=10,random_state=1).fit(Z)
print('Raw / standardized ARI:',adjusted_rand_score(y,raw),adjusted_rand_score(y,model.labels_))
print('Standardized silhouette:',silhouette_score(Z,model.labels_))
assert adjusted_rand_score(y,model.labels_)>.9
assert adjusted_rand_score(y,raw)<.1


Raw / standardized ARI: 0.0020581284066312896 1.0
Standardized silhouette: 0.5330150144947168


In [2]:
stability=[]
for seed in range(12):
    ix=np.random.default_rng(seed).integers(0,len(Z),len(Z))
    fit=KMeans(2,n_init=10,random_state=seed).fit(Z[ix])
    stability.append(adjusted_rand_score(model.labels_,fit.predict(Z)))
print('Bootstrap partition ARI range:',min(stability),max(stability))
assert adjusted_rand_score(model.labels_,1-model.labels_)==1
assert np.mean(stability)>.8


Bootstrap partition ARI range: 1.0 1.0


## Deliberate failure and repair

The raw-distance result clusters a large-unit nuisance measurement. Repair the feature definition and justify scaling. Do not claim that the standardized solution would be correct without the planted truth. The label-swap assertion demonstrates why array equality would report a false failure for the same partition.

## Your investigation

Try K=2 and K=3 and inspect sizes, not only a scalar score. Propose an external variable that could validate a cluster hypothesis without having been used to construct the clusters. Describe how scanner balance and participant-level resampling would enter a real stability analysis.

## Transfer to real neuroimaging

Clustering is a supplementary modeling topic here, not claimed as a named BrainIAK or NIIN class. For real imaging, preserve participant, acquisition, and feature metadata; treat subtype discovery as exploratory until evaluated in independent participants and settings.

**Primary teaching sources, pinned where hosted on GitHub:**

- [NMA: representation and nonlinear visualization](https://github.com/NeuromatchAcademy/course-content/blob/44634e960df7a14cd0bf7398187f2d209d26b0e8/tutorials/W1D4_DimensionalityReduction/student/W1D4_Tutorial4.ipynb)
- [scikit-learn clustering guide](https://scikit-learn.org/stable/modules/clustering.html)

Pinned upstream tutorials are a separate assignment; they have **not been executed** by this core lab. They may require data downloads, specialist dependencies, unfinished student cells, and additional compute.

## Exit questions and answer key

1. Does a stable cluster prove a biological subtype? **No; stable nuisance or an arbitrary cut through a continuum can also reproduce.**
2. Why use a label-invariant score? **Cluster numbers can permute without changing membership.**

### Return to the research question

Reopen [PD03](../../curriculum/papers/design.md#pd03) and your initial two-sentence prediction. In your [evidence ledger](../../curriculum/coursework/EVIDENCE_LEDGER.md):

1. Cite one output or diagnostic from this lesson and explain the transformation it demonstrates.
2. Revise one claim or question from the paper, with a figure/section locator. State what this small exercise still cannot establish about the published result.
3. Ask AI to propose a next check. Accept, revise or reject it with a scientific reason. Then explain your decision aloud without reading the AI response.

Reuse this entry in the A2 portfolio when relevant; a separate report is unnecessary.
